
# 📈 Stock Price Predictor

This Google Colab notebook builds a simple **Stock Price Prediction system using Linear Regression**.

### What this notebook does
1. Downloads historical stock data using `yfinance`
2. Visualizes historical closing prices
3. Creates time-based features
4. Splits the data into training and testing sets
5. Trains a Linear Regression model
6. Evaluates the model using MAE, MSE and R²
7. Plots actual vs predicted prices
8. Predicts the next trading day's closing price
9. Shows a short-term forecast for the next 5 trading days

> **Educational project:** Stock prices are highly unpredictable. The predictions produced here should not be treated as financial advice.


In [ ]:

# Install required libraries
!pip -q install yfinance scikit-learn pandas numpy matplotlib


In [ ]:

# Import libraries
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score



## 1. Select a Stock

You can change the ticker below.

Examples:
- `AAPL` → Apple
- `MSFT` → Microsoft
- `GOOGL` → Alphabet
- `AMZN` → Amazon
- `TSLA` → Tesla
- `RELIANCE.NS` → Reliance Industries (NSE)
- `TCS.NS` → TCS (NSE)
- `INFY.NS` → Infosys (NSE)


In [ ]:

# Stock settings
TICKER = "AAPL"       # Change this ticker if you want
START_DATE = "2018-01-01"
END_DATE = None       # None = download data up to the latest available date

print("Selected stock:", TICKER)


In [ ]:

# Download historical stock data
data = yf.download(
    TICKER,
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False,
    progress=False
)

if data.empty:
    raise ValueError("No stock data was downloaded. Check the ticker symbol and try again.")

# Handle possible MultiIndex columns returned by yfinance
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

data = data.dropna().copy()

print("Rows:", len(data))
print("Date range:", data.index.min().date(), "to", data.index.max().date())
data.tail()


In [ ]:

# Plot historical closing price
plt.figure(figsize=(14, 6))
plt.plot(data.index, data["Close"], label="Closing Price")
plt.title(f"{TICKER} Historical Closing Price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()



## 2. Prepare Features

For a simple and understandable model, we use:

- `Open`
- `High`
- `Low`
- `Volume`
- `Previous_Close`

The target variable is the **next day's closing price**.


In [ ]:

# Create prediction target: next trading day's closing price
data["Previous_Close"] = data["Close"].shift(1)
data["Target"] = data["Close"].shift(-1)

features = ["Open", "High", "Low", "Volume", "Previous_Close"]

model_data = data[features + ["Target"]].dropna().copy()

X = model_data[features]
y = model_data["Target"]

print("Feature columns:", features)
print("Number of usable rows:", len(model_data))
model_data.head()



## 3. Train-Test Split

Because stock data is time-dependent, we **do not randomly shuffle** the data.

The first 80% is used for training and the final 20% is used for testing.


In [ ]:

# Time-series train/test split
split_index = int(len(model_data) * 0.80)

X_train = X.iloc[:split_index]
X_test = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test = y.iloc[split_index:]

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


In [ ]:

# Train Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully.")
print("\nModel coefficients:")
for feature, coefficient in zip(features, model.coef_):
    print(f"{feature:16s}: {coefficient:.6f}")

print(f"Intercept         : {model.intercept_:.6f}")



## 4. Evaluate the Model


In [ ]:

# Make predictions on test data
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Model Performance")
print("------------------")
print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


In [ ]:

# Plot actual vs predicted prices
plt.figure(figsize=(14, 6))
plt.plot(y_test.index, y_test.values, label="Actual Price")
plt.plot(y_test.index, y_pred, label="Predicted Price")
plt.title(f"{TICKER} - Actual vs Predicted Closing Price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:

# Show a sample of predictions
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
}, index=y_test.index)

results["Error"] = results["Actual"] - results["Predicted"]

results.tail(10)



## 5. Predict the Next Trading Day

The latest available day's market data is used to estimate the next trading day's closing price.


In [ ]:

# Prepare latest available input
latest = data.iloc[-1]

latest_features = pd.DataFrame([{
    "Open": float(latest["Open"]),
    "High": float(latest["High"]),
    "Low": float(latest["Low"]),
    "Volume": float(latest["Volume"]),
    "Previous_Close": float(latest["Previous_Close"])
}])

next_day_prediction = model.predict(latest_features)[0]

print("Latest available date:", data.index[-1].date())
print(f"Latest closing price : {float(data['Close'].iloc[-1]):.2f}")
print(f"Predicted next close : {next_day_prediction:.2f}")



## 6. Five-Day Recursive Forecast

The following section creates an **educational short-term forecast**. Since future Open/High/Low/Volume values are unknown, it uses the previous predicted close as a simplified input for future days.

This is only a demonstration of how recursive forecasting works and is **not a reliable trading forecast**.


In [ ]:

# Simple recursive 5-day forecast
last_close = float(data["Close"].iloc[-1])
last_open = float(data["Open"].iloc[-1])
last_high = float(data["High"].iloc[-1])
last_low = float(data["Low"].iloc[-1])
last_volume = float(data["Volume"].iloc[-1])

future_predictions = []

for day in range(1, 6):
    future_input = pd.DataFrame([{
        "Open": last_close,
        "High": last_close,
        "Low": last_close,
        "Volume": last_volume,
        "Previous_Close": last_close
    }])

    predicted_close = float(model.predict(future_input)[0])

    future_predictions.append(predicted_close)
    last_close = predicted_close

forecast_dates = pd.bdate_range(
    start=data.index[-1] + pd.Timedelta(days=1),
    periods=5
)

forecast = pd.DataFrame({
    "Date": forecast_dates,
    "Predicted_Close": future_predictions
})

forecast


In [ ]:

# Plot 5-day forecast
plt.figure(figsize=(12, 6))

recent_data = data.tail(60)

plt.plot(
    recent_data.index,
    recent_data["Close"],
    label="Historical Close"
)

plt.plot(
    forecast["Date"],
    forecast["Predicted_Close"],
    marker="o",
    linestyle="--",
    label="5-Day Forecast"
)

plt.title(f"{TICKER} - Short-Term Stock Price Forecast")
plt.xlabel("Date")
plt.ylabel("Price")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()



## 7. Conclusion

The notebook demonstrates a complete machine-learning workflow for stock price prediction:

**Historical Data → Feature Engineering → Train/Test Split → Linear Regression → Evaluation → Prediction → Forecast**

### Important limitation
A linear regression model using a few market variables cannot capture all factors that affect stock prices, such as news, market sentiment, economic conditions, company announcements, global events, and volatility.

For a more advanced project, you could experiment with:
- Random Forest
- Gradient Boosting
- XGBoost
- LSTM
- Technical indicators such as SMA, EMA, RSI and MACD
- Walk-forward validation
- Sentiment analysis from financial news

**This notebook is intended for learning and demonstration, not financial advice.**
